# 面试问题：Agentic RAG 怎样做多跳问题分解、检索与证据闭环？

可以直接复述的回答是：第一，先识别回答所需的独立事实槽位，再生成子问题。第二，每个子问题单独检索并记录 doc_id、版本和命中理由。第三，最终答案只能组合已覆盖的证据，缺槽位就拒答或继续检索。第四，时间、地区和制度版本必须参与过滤。第五，Baseline 应是整句一次检索，用相同证据覆盖率比较。第六，失败案例要验证旧文档或局部证据不会被拼成确定结论。下面用企业采购制度问答实现完整链路。

## 真实案例：跨金额、品类和地区的采购审批问答

知识库包含 8 条脱敏制度片段，覆盖金额阈值、研发硬件分类、上海审批和旧版制度。五条员工问题带金额、品类、地区与日期，目标是回答是否比价以及谁审批。文档是教学用离线摘录，结构与真实制度库一致；规则不代表任何公司的真实政策。

In [1]:
documents = [  # 定义八条带版本、地区和事实类型的制度证据
    {"id": "D1", "kind": "threshold", "valid_from": "2026-01-01", "valid_to": None, "min": 0, "max": 2000, "text": "单笔采购低于 2000 元无需三方比价，由成本中心负责人确认。"},  # 当前低金额规则
    {"id": "D2", "kind": "threshold", "valid_from": "2026-01-01", "valid_to": None, "min": 2000, "max": 10000, "text": "单笔采购 2000 至 10000 元至少取得三家有效报价。"},  # 当前中金额比价规则
    {"id": "D3", "kind": "threshold", "valid_from": "2026-01-01", "valid_to": None, "min": 10000, "max": 10**9, "text": "单笔采购达到 10000 元需进入集中采购并由采购负责人审批。"},  # 当前高金额集中采购规则
    {"id": "D4", "kind": "category", "category": "研发硬件", "valid_from": "2026-01-01", "valid_to": None, "text": "显示器、GPU 和开发工作站归类为研发硬件。"},  # 品类映射证据
    {"id": "D5", "kind": "region", "region": "上海", "valid_from": "2026-01-01", "valid_to": None, "min": 5000, "text": "上海研发团队采购达到 5000 元还需部门负责人审批。"},  # 上海附加审批规则
    {"id": "D6", "kind": "threshold", "valid_from": "2024-01-01", "valid_to": "2025-12-31", "min": 3000, "max": 12000, "text": "旧制度：采购 3000 至 12000 元只需两家报价。"},  # 故意保留的过期冲突规则
    {"id": "D7", "kind": "category", "category": "软件订阅", "valid_from": "2026-01-01", "valid_to": None, "text": "SaaS 许可证和云软件服务归类为软件订阅。"},  # 软件品类映射证据
    {"id": "D8", "kind": "region", "region": "北京", "valid_from": "2026-01-01", "valid_to": None, "min": 8000, "text": "北京办公室采购达到 8000 元需行政负责人会签。"},  # 北京附加审批规则
]  # 结束小型版本化制度库
questions = [  # 定义五条需要组合多个事实槽位的员工问题
    {"id": "Q1", "text": "上海研发团队采购 9000 元显示器，需要比价吗，谁审批？", "amount": 9000, "category": "研发硬件", "region": "上海", "date": "2026-07-01", "gold": {"D2", "D4", "D5"}},  # 中金额上海硬件问题
    {"id": "Q2", "text": "北京办公室采购 1500 元显示器，需要比价吗？", "amount": 1500, "category": "研发硬件", "region": "北京", "date": "2026-07-01", "gold": {"D1", "D4"}},  # 低金额硬件问题
    {"id": "Q3", "text": "上海研发团队采购 12000 元 GPU，走什么流程？", "amount": 12000, "category": "研发硬件", "region": "上海", "date": "2026-07-01", "gold": {"D3", "D4", "D5"}},  # 高金额集中采购问题
    {"id": "Q4", "text": "北京购买 8500 元 SaaS 许可证，需要哪些审批？", "amount": 8500, "category": "软件订阅", "region": "北京", "date": "2026-07-01", "gold": {"D2", "D7", "D8"}},  # 北京软件订阅问题
    {"id": "Q5", "text": "上海采购 1800 元开发工作站，需要三方报价吗？", "amount": 1800, "category": "研发硬件", "region": "上海", "date": "2026-07-01", "gold": {"D1", "D4"}},  # 低金额上海硬件问题
]  # 结束多跳问答评测集
print("问题输入：id | amount | category | region | question")  # 展示查询规划器接收的结构化上下文
for question in questions:  # 逐条输出五个真实语义问题
    print(f"{question['id']} | {question['amount']:5} | {question['category']:5} | {question['region']} | {question['text']}")  # 呈现金额、品类和地区槽位
print("知识库文档：", [(doc["id"], doc["kind"], doc["valid_from"], doc["valid_to"]) for doc in documents])  # 展示证据版本元数据


问题输入：id | amount | category | region | question
Q1 |  9000 | 研发硬件  | 上海 | 上海研发团队采购 9000 元显示器，需要比价吗，谁审批？
Q2 |  1500 | 研发硬件  | 北京 | 北京办公室采购 1500 元显示器，需要比价吗？
Q3 | 12000 | 研发硬件  | 上海 | 上海研发团队采购 12000 元 GPU，走什么流程？
Q4 |  8500 | 软件订阅  | 北京 | 北京购买 8500 元 SaaS 许可证，需要哪些审批？
Q5 |  1800 | 研发硬件  | 上海 | 上海采购 1800 元开发工作站，需要三方报价吗？
知识库文档： [('D1', 'threshold', '2026-01-01', None), ('D2', 'threshold', '2026-01-01', None), ('D3', 'threshold', '2026-01-01', None), ('D4', 'category', '2026-01-01', None), ('D5', 'region', '2026-01-01', None), ('D6', 'threshold', '2024-01-01', '2025-12-31'), ('D7', 'category', '2026-01-01', None), ('D8', 'region', '2026-01-01', None)]


## Baseline / 基线：整句一次关键词检索

Baseline 直接按文本中重合的汉字二元组排序，并取 Top-2。它可能命中“采购”和金额相关文本，却无法保证阈值、品类和地区三个证据槽位都齐全。

In [2]:
def bigrams(text):  # 使用连续中文字符二元组构造透明关键词特征
    chinese = "".join(character for character in text if "一" <= character <= "鿿")  # 去除数字和标点以保留中文语义序列
    return {chinese[index:index + 2] for index in range(max(0, len(chinese) - 1))}  # 返回去重后的相邻二元组
def baseline_retrieve(question, top_k=2):  # 实现整句一次检索的最小基线
    query_terms = bigrams(question["text"])  # 提取用户问题的中文二元组
    scored = []  # 收集每条制度片段的重合分数
    for document in documents:  # 对版本混合的完整知识库直接排序
        score = len(query_terms & bigrams(document["text"]))  # 计算问题与文档的二元组交集
        scored.append((score, document["id"]))  # 保存可解释的重合数量
    return sorted(scored, key=lambda item: (-item[0], item[1]))[:top_k]  # 返回分数最高的两个证据
baseline_q1 = baseline_retrieve(questions[0])  # 对上海 9000 元显示器问题运行整句检索
print("Q1 单跳 Top-2：score | doc_id")  # 输出基线检索结果
for score, document_id in baseline_q1:  # 展示两个候选片段的重合分数
    print(f"{score} | {document_id}")  # 让证据缺槽问题可被直接观察
print("Q1 所需证据：", sorted(questions[0]["gold"]))  # 展示人工期望的三类证据


Q1 单跳 Top-2：score | doc_id
8 | D5
3 | D4
Q1 所需证据： ['D2', 'D4', 'D5']


## 核心实现：问题分解、有效期过滤与证据账本

规划器把问题拆成金额阈值、品类定义和地区附加规则三个子任务。检索器使用结构化槽位过滤，返回时记录选择理由；没有适用地区规则时不强行补文档。

In [3]:
def is_valid(document, query_date):  # 判断制度片段在查询日期是否有效
    starts = document["valid_from"] <= query_date  # 检查制度已经生效
    not_expired = document["valid_to"] is None or query_date <= document["valid_to"]  # 检查制度尚未过期
    return starts and not_expired  # 只有两个时间条件同时满足才可作为证据
def plan_subquestions(question):  # 根据业务槽位生成可审计多跳计划
    return [("threshold", f"金额 {question['amount']} 对应什么比价阈值"), ("category", f"{question['category']} 如何分类"), ("region", f"{question['region']} 是否有附加审批")]  # 返回三个独立事实目标
def retrieve_evidence(question):  # 为每个子任务选择版本和条件正确的证据
    evidence = []  # 收集 doc_id、子任务和选择理由
    for kind, subquestion in plan_subquestions(question):  # 逐个解决三个事实槽位
        candidates = [document for document in documents if document["kind"] == kind and is_valid(document, question["date"])]  # 先过滤事实类型与有效期
        if kind == "threshold":  # 金额规则按闭开区间匹配
            candidates = [document for document in candidates if document["min"] <= question["amount"] < document["max"]]  # 只保留当前金额所在阈值
        elif kind == "category":  # 品类规则按结构化分类匹配
            candidates = [document for document in candidates if document["category"] == question["category"]]  # 只保留当前采购品类定义
        else:  # 地区规则需要同时满足地区和起始金额
            candidates = [document for document in candidates if document["region"] == question["region"] and question["amount"] >= document["min"]]  # 只保留实际触发的地区附加审批
        for document in candidates:  # 把符合条件的制度片段写入证据账本
            evidence.append({"hop": kind, "subquestion": subquestion, "doc_id": document["id"], "reason": document["text"]})  # 保留子问题、来源和原文理由
    return evidence  # 返回可用于答案组合的完整证据链
evidence_q1 = retrieve_evidence(questions[0])  # 对 Q1 执行三跳检索计划
print("Q1 查询计划与证据：hop | subquestion | doc_id | reason")  # 输出 Agentic RAG 的核心中间过程
for item in evidence_q1:  # 逐跳展示证据来源
    print(f"{item['hop']:9} | {item['subquestion']} | {item['doc_id']} | {item['reason']}")  # 让证据覆盖可人工复核


Q1 查询计划与证据：hop | subquestion | doc_id | reason
threshold | 金额 9000 对应什么比价阈值 | D2 | 单笔采购 2000 至 10000 元至少取得三家有效报价。
category  | 研发硬件 如何分类 | D4 | 显示器、GPU 和开发工作站归类为研发硬件。
region    | 上海 是否有附加审批 | D5 | 上海研发团队采购达到 5000 元还需部门负责人审批。


## 失败案例与修正：过期制度造成冲突答案

旧制度 D6 含有“采购、报价”等高频词，未做时间过滤的检索可能把“两家报价”拼进 2026 年答案。修正后所有候选先通过有效期门禁，再按结构化条件取证。

In [4]:
old_document = next(document for document in documents if document["id"] == "D6")  # 取出与当前规则冲突的旧版制度
naive_old_allowed = "报价" in old_document["text"] and questions[0]["amount"] >= old_document["min"]  # 模拟只看关键词和金额下限的错误选择
safe_old_allowed = is_valid(old_document, questions[0]["date"])  # 使用查询日期验证制度有效期
safe_q1_ids = {item["doc_id"] for item in evidence_q1}  # 汇总修正后 Q1 的证据编号
coverage_q1 = len(safe_q1_ids & questions[0]["gold"]) / len(questions[0]["gold"])  # 计算三类期望证据的覆盖率
print("过期文档：", old_document["id"], old_document["text"])  # 展示可能污染答案的真实冲突文本
print(f"修正前可能采用旧制度={naive_old_allowed}，修正后有效期通过={safe_old_allowed}")  # 对照时间门禁前后行为
print("Q1 安全证据：", sorted(safe_q1_ids), "覆盖率=", f"{coverage_q1:.0%}")  # 展示修正后的完整证据集合


过期文档： D6 旧制度：采购 3000 至 12000 元只需两家报价。
修正前可能采用旧制度=True，修正后有效期通过=False
Q1 安全证据： ['D2', 'D4', 'D5'] 覆盖率= 100%


## 结果表：五条问题的单跳与多跳证据覆盖率

In [5]:
baseline_coverages = []  # 收集整句 Top-2 对人工证据的覆盖率
planned_coverages = []  # 收集多跳规划对人工证据的覆盖率
print("id | gold | baseline_docs | baseline_coverage | planned_docs | planned_coverage")  # 输出逐问题证据对照表
for question in questions:  # 在同一批五条问题上比较两种检索策略
    baseline_ids = {document_id for _, document_id in baseline_retrieve(question)}  # 获取单跳 Top-2 证据编号
    planned_ids = {item["doc_id"] for item in retrieve_evidence(question)}  # 获取多跳结构化证据编号
    baseline_coverage = len(baseline_ids & question["gold"]) / len(question["gold"])  # 计算单跳证据覆盖率
    planned_coverage = len(planned_ids & question["gold"]) / len(question["gold"])  # 计算多跳证据覆盖率
    baseline_coverages.append(baseline_coverage)  # 保存基线指标供汇总
    planned_coverages.append(planned_coverage)  # 保存多跳指标供汇总
    print(f"{question['id']} | {sorted(question['gold'])} | {sorted(baseline_ids)} | {baseline_coverage:.0%} | {sorted(planned_ids)} | {planned_coverage:.0%}")  # 展示每条问题缺失的证据
mean_baseline_coverage = sum(baseline_coverages) / len(baseline_coverages)  # 计算单跳平均证据覆盖率
mean_planned_coverage = sum(planned_coverages) / len(planned_coverages)  # 计算多跳平均证据覆盖率
print(f"平均证据覆盖率：single_query={mean_baseline_coverage:.1%}，multi_hop={mean_planned_coverage:.1%}")  # 输出同指标汇总对照


id | gold | baseline_docs | baseline_coverage | planned_docs | planned_coverage
Q1 | ['D2', 'D4', 'D5'] | ['D4', 'D5'] | 67% | ['D2', 'D4', 'D5'] | 100%
Q2 | ['D1', 'D4'] | ['D1', 'D8'] | 50% | ['D1', 'D4'] | 100%
Q3 | ['D3', 'D4', 'D5'] | ['D1', 'D5'] | 33% | ['D3', 'D4', 'D5'] | 100%
Q4 | ['D2', 'D7', 'D8'] | ['D3', 'D7'] | 33% | ['D2', 'D7', 'D8'] | 100%
Q5 | ['D1', 'D4'] | ['D1', 'D4'] | 100% | ['D1', 'D4'] | 100%
平均证据覆盖率：single_query=56.7%，multi_hop=100.0%


## 结果解读

Q1 必须同时获得 D2 的比价阈值、D4 的品类定义和 D5 的上海附加审批；整句 Top-2 天生无法覆盖三个证据槽。多跳计划把“为什么需要三家报价”和“谁额外审批”分别绑定来源，并排除已于 2025 年失效的 D6。证据覆盖率衡量的是取证完整度，不等于答案语言质量。

## 生产边界

生产 RAG 还需要真实倒排或向量索引、文档级 ACL、版本时间语义、冲突检测、引用片段定位、查询预算和无证据拒答。结构化金额和地区可能来自实体抽取，需要单独评测；制度更新时索引与缓存必须原子切换。本例没有评估生成模型幻觉和长文档切块。

## 最小回归测试

In [6]:
assert len(questions) >= 5  # 保证案例至少包含五条真实采购问题
assert safe_q1_ids == questions[0]["gold"]  # 保证 Q1 的三类证据全部且准确命中
assert safe_old_allowed is False  # 保证 2024 旧制度不能进入 2026 年答案
assert coverage_q1 == 1.0  # 保证核心示例达到完整证据覆盖
assert mean_planned_coverage > mean_baseline_coverage  # 保证多跳规划在同一评测集上优于整句 Top-2
